In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI()
print("Setup successful! Ready to go.")

Setup successful! Ready to go.


In [3]:
# STATIC PROMPT - hardcoded, always the same
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is photosynthesis?"}],
    temperature=0
)

print(response.choices[0].message.content)

Photosynthesis is a biochemical process used by plants, algae, and some bacteria to convert light energy, usually from the sun, into chemical energy stored in glucose (a type of sugar). This process is essential for life on Earth, as it provides the primary source of energy for nearly all living organisms, either directly or indirectly.

The general equation for photosynthesis can be summarized as follows:

\[ 6 \, \text{CO}_2 + 6 \, \text{H}_2\text{O} + \text{light energy} \rightarrow \text{C}_6\text{H}_{12}\text{O}_6 + 6 \, \text{O}_2 \]

In this equation:
- \( \text{CO}_2 \) (carbon dioxide) is absorbed from the atmosphere through small openings in leaves called stomata.
- \( \text{H}_2\text{O} \) (water) is taken up by the roots from the soil.
- Light energy is captured by chlorophyll, the green pigment in chloroplasts, which are specialized organelles in plant cells.

Photosynthesis occurs in two main stages:

1. **Light-dependent reactions**: These occur in the thylakoid membrane

In [4]:
# If we want to ask a different question, we have to CHANGE THE CODE
# This is bad because we can't reuse the code

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What is machine learning?"}],  # CHANGED THIS
    temperature=0
)

print(response.choices[0].message.content)

Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform tasks without explicit programming. Instead of being programmed with specific instructions for every possible scenario, machine learning systems learn from data, identify patterns, and make decisions based on that data.

Key concepts in machine learning include:

1. **Data**: Machine learning relies on large amounts of data to train models. This data can be structured (like databases) or unstructured (like text, images, or audio).

2. **Algorithms**: These are the mathematical models and techniques used to analyze data and make predictions. Common algorithms include decision trees, neural networks, support vector machines, and clustering algorithms.

3. **Training**: The process of feeding data into a machine learning model so that it can learn from it. During training, the model adjusts its parameters to minimize errors in i

In [5]:
# DYNAMIC PROMPT - uses a variable
user_question = "What is machine learning?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": f"Answer this question concisely: {user_question}"}],
    temperature=0
)

print(response.choices[0].message.content)

Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from and make predictions or decisions based on data, without being explicitly programmed for specific tasks.


In [6]:
# SAME CODE, DIFFERENT QUESTION - no code changes needed
user_question = "What is the capital of France?"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": f"Answer this question concisely: {user_question}"}],
    temperature=0
)

print(response.choices[0].message.content)

The capital of France is Paris.


In [7]:
def ask(question):
    """Send a dynamic question to the model."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Answer this question concisely: {question}"}],
        temperature=0
    )
    return response.choices[0].message.content

# Now we can ask ANY question with the same function
print(ask("What is an API?"))
print("---")
print(ask("Explain recursion in one sentence."))

An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and interact with each other. It defines the methods and data formats that applications can use to request and exchange information.
---
Recursion is a programming technique where a function calls itself to solve smaller instances of the same problem until a base condition is met.


In [8]:
def ask_with_style(question, tone="professional", max_sentences=2):
    """Send a question with configurable tone and length."""
    prompt = (
        f"Answer the following question in a {tone} tone. "
        f"Keep your response to {max_sentences} sentences maximum.\n\n"
        f"Question: {question}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# Same question, different styles
print("Professional:")
print(ask_with_style("What is cloud computing?", tone="professional"))
print("\nCasual:")
print(ask_with_style("What is cloud computing?", tone="casual and friendly"))

Professional:
Cloud computing is the delivery of computing services—including storage, processing power, and applications—over the internet, allowing users to access and manage resources remotely. This model enables scalability, flexibility, and cost-efficiency for businesses and individuals alike.

Casual:
Cloud computing is like storing and accessing your files and apps over the internet instead of on your computer's hard drive. It makes it super easy to access your stuff from anywhere, anytime!


In [3]:
# THIS IS HOW RAG WORKS (simplified version)

def answer_with_context(question, context):
    """Answer a question using provided context."""
    prompt = (
        f"Use the following context to answer the question. "
        f"If the context doesn't contain the answer, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# Company-specific info the model was never trained on
company_context = """
Acme Corp was founded in 2019. The CEO is Jane Smith.
Acme Corp has 150 employees and is headquartered in Berlin.
Their main product is an AI-powered inventory management system.
"""

print(answer_with_context("Who is the CEO of Acme Corp?", company_context))
print("---")
print(answer_with_context("What does Acme Corp sell?", company_context))

The CEO of Acme Corp is Jane Smith.
---
Acme Corp sells an AI-powered inventory management system.


In [4]:
def chat_with_memory():
    """Simple chat loop that maintains conversation history."""
    messages = [{"role": "system", "content": "You are a helpful assistant. Be concise."}]

    questions = [
        "What is Python?",
        "What are its main use cases?",  # 'its' refers to Python — the model needs memory
        "Which one is best for beginners?"
    ]

    for question in questions:
        messages.append({"role": "user", "content": question})

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0
        )

        answer = response.choices[0].message.content
        messages.append({"role": "assistant", "content": answer})

        print(f"User: {question}")
        print(f"Assistant: {answer}\n")

chat_with_memory()

User: What is Python?
Assistant: Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, automation, and more. Its extensive standard library and active community contribute to its popularity.

User: What are its main use cases?
Assistant: Python's main use cases include:

1. **Web Development**: Frameworks like Django and Flask facilitate building web applications.
2. **Data Analysis and Visualization**: Libraries like Pandas, NumPy, and Matplotlib are used for data manipulation and visualization.
3. **Machine Learning and AI**: Libraries such as TensorFlow, Keras, and Scikit-learn support machine learning and deep learning projects.
4. **Automation and Scripting**: Python is often used for automating repetitive tasks and writing 